<a href="https://colab.research.google.com/github/clee2026/MSDS_498/blob/main/capstone/finding_large/notebook_3_full_analysis_modeling_20M_standard_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Notebook 3 - Full Analysis and Modeling  

## Required Inputs

Before running this notebook, upload the following file into Colab:

- notebook2_outputs.zip

This zip file should come from Notebook 2 and should contain:

- analysis_ready_sample.parquet
- final_field_list.csv
- missing_value_treatment.csv
- derived_features_summary.csv
- transformation_rules_applied.csv
- run_manifest_notebook2.json
- notebook2_completion_summary.txt

## Additional Requirement

The full original parquet dataset must still be available in Google Drive.

Notebook 3 uses:

- **Notebook 2 sample parquet** for modeling
- **Full original parquet** for batch aggregation

This avoids loading all 20 million rows into memory.

## If the Drive parquet path fails

The manifest contains the original parquet path. If the path is not found, set this variable manually in the path resolution cell:

```python
MANUAL_SOURCE_PARQUET = "/content/drive/MyDrive/your_actual_path_here.parquet"
```

## Outputs Generated

This notebook creates:

- descriptive summary tables
- full-data batch aggregation outputs
- statistical analysis outputs
- regression model results
- classification model results
- feature importance outputs
- plots for reporting
- notebook3_outputs.zip



In [ ]:
# 1. Environment Setup

import os
import json
import gc
import shutil
import zipfile
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pyarrow.parquet as pq

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive



## Unzip Notebook 2 outputs

This section unzips notebook2_outputs.zip and loads the metadata and prepared sample dataset.


In [ ]:
# 2. Unzip Notebook 2 Outputs

ZIP_PATH = "/content/notebook2_outputs.zip"
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError("Upload notebook2_outputs.zip to /content before running this notebook.")

N2_DIR = "/content/notebook2_outputs"
os.makedirs(N2_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(N2_DIR)

print("Notebook 2 files extracted:")
for p in sorted(os.listdir(N2_DIR)):
    print(" -", p)


Notebook 2 files extracted:
 - analysis_ready_sample.parquet
 - derived_features_summary.csv
 - final_field_list.csv
 - missing_value_treatment.csv
 - notebook2_completion_summary.txt
 - run_manifest_notebook2.json
 - transformation_rules_applied.csv


In [ ]:
# 3. Load Notebook 2 Handoff Files

final_field_list = pd.read_csv(f"{N2_DIR}/final_field_list.csv")
missing_value_treatment = pd.read_csv(f"{N2_DIR}/missing_value_treatment.csv")
derived_features_summary = pd.read_csv(f"{N2_DIR}/derived_features_summary.csv")
transformation_rules_applied = pd.read_csv(f"{N2_DIR}/transformation_rules_applied.csv")

manifest_path = f"{N2_DIR}/run_manifest_notebook2.json"
with open(manifest_path, "r") as f:
    manifest = json.load(f)

completion_summary_path = f"{N2_DIR}/notebook2_completion_summary.txt"
if os.path.exists(completion_summary_path):
    with open(completion_summary_path, "r") as f:
        notebook2_summary = f.read()
else:
    notebook2_summary = ""

print("Loaded Notebook 2 handoff files.")
display(final_field_list.head())
display(derived_features_summary)


Loaded Notebook 2 handoff files.


,column_name
0,descriptor_2
1,landmark
2,intersection_street_1
3,intersection_street_2
4,cross_street_2


,derived_feature,source_columns,purpose
0,response_time_hours,"created_date, closed_date",Regression target for service response time
1,closed_same_day_flag,"created_date, closed_date",Classification target for same-day resolution
2,repeat_complaint_flag,complaint_type,Exploratory classification flag for repeated c...



## Create Notebook 3 output folders

All local outputs are saved under /content/project_outputs/03_analysis and zipped at the end.


In [ ]:
# 4. Prepare Output Folders

OUTPUT_DIR = manifest.get("output_dir", "/content/project_outputs")
ANALYSIS_DIR = f"{OUTPUT_DIR}/03_analysis"
TABLE_DIR = f"{ANALYSIS_DIR}/tables"
PLOT_DIR = f"{ANALYSIS_DIR}/plots"
MODEL_DIR = f"{ANALYSIS_DIR}/models"

for d in [ANALYSIS_DIR, TABLE_DIR, PLOT_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print("Notebook 3 output directory:", ANALYSIS_DIR)


Notebook 3 output directory: /content/project_outputs/03_analysis



## Resolve the full parquet path

The full parquet is used only for batch aggregation.  
It is not loaded into pandas all at once.

If the manifest path does not exist, set MANUAL_SOURCE_PARQUET below.


In [ ]:
# 5. Resolve Full Source Parquet Path

MANUAL_SOURCE_PARQUET = None
# Example:
# MANUAL_SOURCE_PARQUET = "/content/drive/MyDrive/project_data/Capstone Course Project Files/Data/nyc_311_FINAL_UPDATED.parquet"

SOURCE_PARQUET = MANUAL_SOURCE_PARQUET or manifest.get("source_parquet")

if SOURCE_PARQUET is None:
    raise ValueError("No source parquet path found. Set MANUAL_SOURCE_PARQUET manually.")

if not os.path.exists(SOURCE_PARQUET):
    print("Source parquet path not found:")
    print(SOURCE_PARQUET)
    print("\nSet MANUAL_SOURCE_PARQUET in this cell to the correct Drive path, then rerun this cell.")
    raise FileNotFoundError("Full source parquet was not found.")

print("Full source parquet found:")
print(SOURCE_PARQUET)


Full source parquet found:
/content/drive/MyDrive/project_data/Capstone Course Project Files/Data/nyc_311_FINAL_UPDATED.parquet



## Load the prepared sample dataset

This sample comes from Notebook 2 and is used for modeling, plots, and quick statistical analysis.


In [ ]:
# 6. Load Analysis-Ready Sample

sample_path = f"{N2_DIR}/analysis_ready_sample.parquet"
if not os.path.exists(sample_path):
    raise FileNotFoundError("analysis_ready_sample.parquet was not found in notebook2_outputs.zip.")

df_sample = pd.read_parquet(sample_path)

print("Sample shape:", df_sample.shape)
display(df_sample.head())
print(df_sample.dtypes)


Sample shape: (150000, 39)


,descriptor_2,landmark,intersection_street_1,intersection_street_2,cross_street_2,cross_street_1,address_type,location_type,city,street_name,...,longitude,x_coordinate_state_plane,y_coordinate_state_plane,incident_zip,resolution_action_updated_date,created_date,unique_key,response_time_hours,closed_same_day_flag,repeat_complaint_flag
0,UNKNOWN,DE REIMER AVENUE,EDENWALD AVENUE,BUSSING AVENUE,BUSSING AVENUE,EDENWALD AVENUE,ADDRESS,Residential Building/House,BRONX,DEREIMER AVENUE,...,-73.84341190767061,1027542.0,265307.0,10466.0,2026-04-03 20:38:32,2026-04-03 15:05:33,68557295,5.548611,1,1
1,Street,132 STREET,VAN SICLEN STREET,131 STREET,131 STREET,VAN SICLEN STREET,ADDRESS,Street,SOUTH RICHMOND HILL,132 STREET,...,-73.81337953995205,1036005.0,190247.0,11419.0,2026-03-30 10:27:37,2026-03-30 09:17:48,68502105,1.162778,1,1
2,BROKEN OR MISSING,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,ADDRESS,RESIDENTIAL BUILDING,NEW YORK,WEST 204 STREET,...,-73.92172676858291,1005900.0,254371.0,10034.0,2026-03-24 00:00:00,2026-03-24 09:59:46,68445209,NaN,0,1
3,UNKNOWN,GRIMSBY STREET,GREELEY AVENUE,DEAD END,DEAD END,GREELEY AVENUE,ADDRESS,Street,STATEN ISLAND,GRIMSBY STREET,...,-74.09739427872815,957192.0,147156.0,10306.0,2025-12-27 08:35:06,2025-12-27 08:11:04,67313553,0.399722,1,1
4,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,EDENWALD AVENUE,STRANG AVENUE,ADDRESS,Street,BRONX,GRACE AVENUE,...,-73.84518740343698,1027053.0,264247.0,10466.0,2026-03-19 12:00:00,2026-03-18 17:58:00,68379487,10.783333,0,1


descriptor_2                      string[python]
landmark                          string[python]
intersection_street_1             string[python]
intersection_street_2             string[python]
cross_street_2                    string[python]
cross_street_1                    string[python]
address_type                      string[python]
location_type                     string[python]
city                              string[python]
street_name                       string[python]
resolution_description            string[python]
descriptor                        string[python]
council_district                  string[python]
source_file                       string[python]
source_parquet                    string[python]
borough                           string[python]
community_board                   string[python]
park_borough                      string[python]
park_facility_name                string[python]
agency                            string[python]
agency_name         


## Add time features to the modeling sample

These features help models capture time-related patterns without using the raw datetime values directly.


In [ ]:
# 7. Add Sample-Level Time Features

if "created_date" in df_sample.columns:
    df_sample["created_date"] = pd.to_datetime(df_sample["created_date"], errors="coerce")
    df_sample["created_hour"] = df_sample["created_date"].dt.hour
    df_sample["created_dayofweek"] = df_sample["created_date"].dt.dayofweek
    df_sample["created_month"] = df_sample["created_date"].dt.month

if "closed_date" in df_sample.columns:
    df_sample["closed_date"] = pd.to_datetime(df_sample["closed_date"], errors="coerce")

print("Time features added where possible.")


Time features added where possible.



# Full Dataset Batch Aggregation

This section computes high-level summary tables from the full 20M-row parquet without loading the full dataset into pandas.

These outputs support the report's descriptive analysis section.


In [ ]:
# 8. Helper Functions for Full Dataset Batch Aggregation

def clean_name(name: str) -> str:
    name = str(name).strip().lower()
    for old, new in [
        (" ", "_"), ("/", "_"), ("-", "_"), ("(", ""), (")", ""),
        (".", ""), (",", ""), ("&", "and"), ("__", "_")
    ]:
        name = name.replace(old, new)
    while "__" in name:
        name = name.replace("__", "_")
    return name.strip("_")

def build_schema_name_map(parquet_path):
    pf = pq.ParquetFile(parquet_path)
    raw_names = pf.schema.names
    cleaned_to_raw = {clean_name(c): c for c in raw_names}
    return pf, cleaned_to_raw

pf_full, cleaned_to_raw = build_schema_name_map(SOURCE_PARQUET)

def raw_col(cleaned_col):
    return cleaned_to_raw.get(cleaned_col)

print("Full parquet rows:", pf_full.metadata.num_rows)
print("Full parquet columns:", len(pf_full.schema.names))


Full parquet rows: 20774822
Full parquet columns: 46


In [ ]:
# 9. Full Dataset Counts by Key Categories

def full_value_counts(parquet_path, cleaned_col, top_n=25, batch_size=100_000):
    pf, mapping = build_schema_name_map(parquet_path)
    source_col = mapping.get(cleaned_col)

    if source_col is None:
        print(f"Column not found in full parquet: {cleaned_col}")
        return pd.DataFrame(columns=[cleaned_col, "count"])

    counts = Counter()

    for batch in pf.iter_batches(batch_size=batch_size, columns=[source_col]):
        chunk = batch.to_pandas()
        vals = chunk[source_col].fillna("UNKNOWN").astype(str).str.strip()
        counts.update(vals.tolist())
        del chunk, batch
        gc.collect()

    out = pd.DataFrame(counts.items(), columns=[cleaned_col, "count"])
    out = out.sort_values("count", ascending=False).head(top_n)
    return out

full_count_tables = {}

for col in ["complaint_type", "borough", "agency", "open_data_channel_type", "status"]:
    tbl = full_value_counts(SOURCE_PARQUET, col, top_n=25)
    if len(tbl) > 0:
        out_path = f"{TABLE_DIR}/full_counts_{col}.csv"
        tbl.to_csv(out_path, index=False)
        full_count_tables[col] = tbl
        print("Saved:", out_path)
        display(tbl.head())


Saved: /content/project_outputs/03_analysis/tables/full_counts_complaint_type.csv


,complaint_type,count
3,Illegal Parking,2641208
2,Noise - Residential,2383077
33,HEAT/HOT WATER,1615888
4,Noise - Street/Sidewalk,1048959
5,Blocked Driveway,992017


Saved: /content/project_outputs/03_analysis/tables/full_counts_borough.csv


,borough,count
2,BROOKLYN,6206211
0,QUEENS,4971263
4,BRONX,4449071
1,MANHATTAN,4194292
3,STATEN ISLAND,876906


Saved: /content/project_outputs/03_analysis/tables/full_counts_agency.csv


,agency,count
2,NYPD,9027208
8,HPD,4173361
1,DSNY,2373775
0,DOT,1384554
9,DEP,1084598


Saved: /content/project_outputs/03_analysis/tables/full_counts_open_data_channel_type.csv


,open_data_channel_type,count
3,ONLINE,8510917
1,PHONE,6320649
2,MOBILE,4158343
0,UNKNOWN,1783538
4,OTHER,1375


Saved: /content/project_outputs/03_analysis/tables/full_counts_status.csv


,status,count
1,Closed,20323070
2,In Progress,235371
0,Open,118610
5,Pending,61067
4,Assigned,26581


In [ ]:
# 10. Plot Full Dataset Top Counts

for col, tbl in full_count_tables.items():
    if len(tbl) == 0:
        continue

    plt.figure(figsize=(10, 6))
    plot_tbl = tbl.head(15).sort_values("count", ascending=True)
    plt.barh(plot_tbl[col].astype(str), plot_tbl["count"])
    plt.title(f"Top {col.replace('_', ' ').title()} Values - Full Dataset")
    plt.xlabel("Count")
    plt.tight_layout()

    out_path = f"{PLOT_DIR}/full_counts_{col}.png"
    plt.savefig(out_path, dpi=150)
    plt.close()
    print("Saved:", out_path)


Saved: /content/project_outputs/03_analysis/plots/full_counts_complaint_type.png
Saved: /content/project_outputs/03_analysis/plots/full_counts_borough.png
Saved: /content/project_outputs/03_analysis/plots/full_counts_agency.png
Saved: /content/project_outputs/03_analysis/plots/full_counts_open_data_channel_type.png
Saved: /content/project_outputs/03_analysis/plots/full_counts_status.png



## Full dataset time trend

This batch process counts requests by month using the full parquet.


In [ ]:
# 11. Full Dataset Monthly Trend

def full_monthly_counts(parquet_path, cleaned_date_col="created_date", batch_size=100_000):
    pf, mapping = build_schema_name_map(parquet_path)
    source_col = mapping.get(cleaned_date_col)

    if source_col is None:
        print(f"Column not found in full parquet: {cleaned_date_col}")
        return pd.DataFrame(columns=["month", "count"])

    counter = Counter()

    for batch in pf.iter_batches(batch_size=batch_size, columns=[source_col]):
        chunk = batch.to_pandas()
        dates = pd.to_datetime(chunk[source_col], errors="coerce")
        months = dates.dt.to_period("M").astype(str)
        counter.update(months.dropna().tolist())
        del chunk, batch, dates, months
        gc.collect()

    out = pd.DataFrame(counter.items(), columns=["month", "count"]).sort_values("month")
    out = out[out["month"] != "NaT"]
    return out

monthly_counts = full_monthly_counts(SOURCE_PARQUET)
monthly_counts.to_csv(f"{TABLE_DIR}/full_monthly_counts.csv", index=False)
display(monthly_counts.head())

if len(monthly_counts) > 0:
    plt.figure(figsize=(12, 5))
    plt.plot(monthly_counts["month"], monthly_counts["count"], marker="o")
    plt.xticks(rotation=90)
    plt.title("Monthly 311 Request Volume - Full Dataset")
    plt.xlabel("Month")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/full_monthly_counts.png", dpi=150)
    plt.close()


,month,count
75,2020-01,195230
74,2020-02,173162
73,2020-03,179779
72,2020-04,159114
71,2020-05,234308



# Sample-Based Descriptive and Statistical Analysis

The sample is used for analysis that requires engineered variables from Notebook 2, such as response_time_hours.


In [ ]:
# 12. Sample Descriptive Tables

summary_tables = {}

if "response_time_hours" in df_sample.columns:
    rt = df_sample["response_time_hours"].replace([np.inf, -np.inf], np.nan).dropna()
    rt = rt[rt >= 0]
    response_summary = pd.DataFrame({
        "metric": ["count", "mean", "median", "std", "min", "p25", "p75", "p90", "p95", "max"],
        "value": [
            rt.count(),
            rt.mean(),
            rt.median(),
            rt.std(),
            rt.min(),
            rt.quantile(0.25),
            rt.quantile(0.75),
            rt.quantile(0.90),
            rt.quantile(0.95),
            rt.max()
        ]
    })
    response_summary.to_csv(f"{TABLE_DIR}/sample_response_time_summary.csv", index=False)
    summary_tables["response_time_summary"] = response_summary
    display(response_summary)

if "complaint_type" in df_sample.columns and "response_time_hours" in df_sample.columns:
    by_complaint = (
        df_sample.assign(response_time_hours=pd.to_numeric(df_sample["response_time_hours"], errors="coerce"))
        .query("response_time_hours >= 0")
        .groupby("complaint_type")["response_time_hours"]
        .agg(["count", "mean", "median"])
        .sort_values("count", ascending=False)
        .head(25)
        .reset_index()
    )
    by_complaint.to_csv(f"{TABLE_DIR}/sample_response_by_complaint_type.csv", index=False)
    summary_tables["response_by_complaint"] = by_complaint
    display(by_complaint.head())


,metric,value
0,count,127611.000000
1,mean,61.091656
2,median,4.599167
3,std,198.206907
4,min,0.000000
5,p25,0.953889
6,p75,44.449861
7,p90,118.967778
8,p95,235.825556
9,max,2301.354167


,complaint_type,count,mean,median
0,Illegal Parking,23430,2.540959,1.459306
1,HEAT/HOT WATER,18259,45.767990,43.551389
2,Noise - Residential,13839,2.235264,1.188333
3,Blocked Driveway,7109,2.671439,1.568611
4,Street Condition,6680,48.258113,15.149306


In [ ]:
# 13. Sample Plots

if "response_time_hours" in df_sample.columns:
    rt = pd.to_numeric(df_sample["response_time_hours"], errors="coerce")
    rt = rt.replace([np.inf, -np.inf], np.nan).dropna()
    rt = rt[(rt >= 0) & (rt <= rt.quantile(0.99))]

    plt.figure(figsize=(9, 5))
    plt.hist(rt, bins=50)
    plt.title("Response Time Distribution - Sample, 99th Percentile Capped")
    plt.xlabel("Response Time Hours")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/sample_response_time_distribution.png", dpi=150)
    plt.close()

if "closed_same_day_flag" in df_sample.columns:
    tbl = df_sample["closed_same_day_flag"].value_counts(dropna=False).reset_index()
    tbl.columns = ["closed_same_day_flag", "count"]
    tbl.to_csv(f"{TABLE_DIR}/sample_closed_same_day_counts.csv", index=False)

    plt.figure(figsize=(6, 4))
    plt.bar(tbl["closed_same_day_flag"].astype(str), tbl["count"])
    plt.title("Closed Same Day Flag Distribution - Sample")
    plt.xlabel("Closed Same Day Flag")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/sample_closed_same_day_counts.png", dpi=150)
    plt.close()



# Modeling Setup

Models are trained on the Notebook 2 sample, not the full 20M rows.

This keeps modeling managable in Colab while preserving full-data summary analysis through batch aggregation.


In [ ]:
import warnings

# Suppress specific sklearn warnings if they become too noisy after the fix
warnings.filterwarnings("ignore", category=UserWarning, module='sklearn')

# 14. Modeling Helpers

def select_model_features(df, target, max_categorical=8):
    excluded = {
        target,
        "response_time_hours",
        "closed_same_day_flag",
        "repeat_complaint_flag",
        "closed_date",
        "resolution_action_updated_date",
        "unique_key",
        "latitude",
        "longitude",
        "x_coordinate_state_plane",
        "y_coordinate_state_plane",
        "bbl",
        "incident_address",
        "resolution_description",
        "source_parquet",
        "source_file",
    }

    preferred = [
        "complaint_type",
        "descriptor",
        "borough",
        "agency",
        "agency_name",
        "location_type",
        "address_type",
        "open_data_channel_type",
        "community_board",
        "incident_zip",
        "police_precinct",
        "created_hour",
        "created_dayofweek",
        "created_month",
        "council_district",
    ]

    features = [c for c in preferred if c in df.columns and c not in excluded]
    features = [c for c in features if df[c].nunique(dropna=False) > 1]

    return features

def build_preprocessor(X):
    # Include 'string' dtype for categorical features to handle newer pandas string type
    categorical_features = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numeric_features = [c for c in X.columns if c not in categorical_features]

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=30))
    ])

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("categorical", categorical_transformer, categorical_features),
            ("numeric", numeric_transformer, numeric_features)
        ],
        remainder="drop"
    )

    return preprocessor, categorical_features, numeric_features

def get_feature_names(preprocessor, categorical_features, numeric_features):
    names = []
    try:
        cat_encoder = preprocessor.named_transformers_["categorical"].named_steps["onehot"]
        # Use original categorical_features list to get feature names out
        cat_names = cat_encoder.get_feature_names_out(categorical_features).tolist()
        names.extend(cat_names)
    except Exception:
        # Fallback if onehot encoder is not present or fails
        names.extend(categorical_features)

    # Only add numeric features that were actually passed to the numeric transformer
    # This filters out numeric features dropped by remainder='drop' if they were not explicitly in numeric_features
    names.extend(numeric_features)
    return names



## Regression Modeling

Target: response_time_hours

Models:
- Dummy baseline
- Ridge regression
- Random Forest regressor

The target is capped at the 99th percentile to reduce extreme long-tail distortion.


In [ ]:
# 15. Regression Modeling: Predict Response Time

regression_results = []
regression_feature_importance = pd.DataFrame()

if "response_time_hours" in df_sample.columns:
    model_df = df_sample.copy()
    model_df["response_time_hours"] = pd.to_numeric(model_df["response_time_hours"], errors="coerce")
    model_df = model_df.replace([np.inf, -np.inf], np.nan)
    model_df = model_df.dropna(subset=["response_time_hours"])
    model_df = model_df[model_df["response_time_hours"] >= 0]

    if len(model_df) > 0:
        cap = model_df["response_time_hours"].quantile(0.99)
        model_df = model_df[model_df["response_time_hours"] <= cap]

        # Keep modeling manageable in standard Colab
        max_model_rows = min(100_000, len(model_df))
        model_df = model_df.sample(n=max_model_rows, random_state=42) if len(model_df) > max_model_rows else model_df

        features = select_model_features(model_df, "response_time_hours")
        X = model_df[features]
        y = model_df["response_time_hours"]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42
        )

        preprocessor, cat_features, num_features = build_preprocessor(X_train)

        models = {
            "dummy_mean": DummyRegressor(strategy="mean"),
            "ridge_regression": Ridge(alpha=1.0),
            "random_forest_regressor": RandomForestRegressor(
                n_estimators=80,
                max_depth=14,
                min_samples_leaf=20,
                random_state=42,
                n_jobs=-1
            )
        }

        fitted_regression_models = {}

        for name, estimator in models.items():
            pipe = Pipeline(steps=[
                ("preprocessor", preprocessor),
                ("model", estimator)
            ])

            pipe.fit(X_train, y_train)
            preds = pipe.predict(X_test)

            rmse = np.sqrt(mean_squared_error(y_test, preds))
            mae = mean_absolute_error(y_test, preds)
            r2 = r2_score(y_test, preds)

            regression_results.append({
                "model": name,
                "target": "response_time_hours",
                "rows_used": len(model_df),
                "features_used": ", ".join(features),
                "rmse": rmse,
                "mae": mae,
                "r2": r2
            })

            fitted_regression_models[name] = pipe
            print(name, "RMSE:", rmse, "MAE:", mae, "R2:", r2)

        regression_results_df = pd.DataFrame(regression_results)
        regression_results_df.to_csv(f"{TABLE_DIR}/regression_model_metrics.csv", index=False)
        display(regression_results_df)

        # Feature importance from random forest
        rf_pipe = fitted_regression_models.get("random_forest_regressor")
        if rf_pipe is not None:
            feature_names = get_feature_names(
                rf_pipe.named_steps["preprocessor"],
                cat_features,
                num_features
            )
            importances = rf_pipe.named_steps["model"].feature_importances_
            regression_feature_importance = (
                pd.DataFrame({"feature": feature_names[:len(importances)], "importance": importances})
                .sort_values("importance", ascending=False)
                .head(30)
            )
            regression_feature_importance.to_csv(f"{TABLE_DIR}/regression_feature_importance.csv", index=False)

            plt.figure(figsize=(10, 7))
            plot_df = regression_feature_importance.sort_values("importance", ascending=True).tail(20)
            plt.barh(plot_df["feature"], plot_df["importance"])
            plt.title("Regression Feature Importance - Response Time")
            plt.tight_layout()
            plt.savefig(f"{PLOT_DIR}/regression_feature_importance.png", dpi=150)
            plt.close()
    else:
        print("No valid rows available for response_time_hours regression.")
else:
    print("response_time_hours not found. Skipping regression.")


dummy_mean RMSE: 114.31475984741894 MAE: 56.43029373982075 R2: -0.0001499693645861111
ridge_regression RMSE: 94.24493310737654 MAE: 38.1652508004878 R2: 0.3202072470616678
random_forest_regressor RMSE: 87.88875114682854 MAE: 32.57365874485057 R2: 0.4088099770688619


,model,target,rows_used,features_used,rmse,mae,r2
0,dummy_mean,response_time_hours,100000,"complaint_type, descriptor, borough, agency, a...",114.314760,56.430294,-0.000150
1,ridge_regression,response_time_hours,100000,"complaint_type, descriptor, borough, agency, a...",94.244933,38.165251,0.320207
2,random_forest_regressor,response_time_hours,100000,"complaint_type, descriptor, borough, agency, a...",87.888751,32.573659,0.408810



## Classification Modeling

Target: closed_same_day_flag

Models:
- Dummy baseline
- Logistic regression
- Random Forest classifier


In [ ]:
# 16. Classification Modeling: Predict Same-Day Closure

classification_results = []
classification_feature_importance = pd.DataFrame()

if "closed_same_day_flag" in df_sample.columns:
    model_df = df_sample.copy()
    model_df["closed_same_day_flag"] = pd.to_numeric(model_df["closed_same_day_flag"], errors="coerce")
    model_df = model_df.dropna(subset=["closed_same_day_flag"])
    model_df["closed_same_day_flag"] = model_df["closed_same_day_flag"].astype(int)

    if model_df["closed_same_day_flag"].nunique() >= 2:
        max_model_rows = min(100_000, len(model_df))
        model_df = model_df.sample(n=max_model_rows, random_state=42) if len(model_df) > max_model_rows else model_df

        features = select_model_features(model_df, "closed_same_day_flag")
        X = model_df[features]
        y = model_df["closed_same_day_flag"]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )

        preprocessor, cat_features, num_features = build_preprocessor(X_train)

        models = {
            "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
            "logistic_regression": LogisticRegression(max_iter=300, n_jobs=-1),
            "random_forest_classifier": RandomForestClassifier(
                n_estimators=100,
                max_depth=14,
                min_samples_leaf=20,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced_subsample"
            )
        }

        fitted_classification_models = {}

        for name, estimator in models.items():
            pipe = Pipeline(steps=[
                ("preprocessor", preprocessor),
                ("model", estimator)
            ])

            pipe.fit(X_train, y_train)
            preds = pipe.predict(X_test)

            acc = accuracy_score(y_test, preds)
            prec = precision_score(y_test, preds, zero_division=0)
            rec = recall_score(y_test, preds, zero_division=0)
            f1 = f1_score(y_test, preds, zero_division=0)

            classification_results.append({
                "model": name,
                "target": "closed_same_day_flag",
                "rows_used": len(model_df),
                "features_used": ", ".join(features),
                "accuracy": acc,
                "precision": prec,
                "recall": rec,
                "f1": f1
            })

            fitted_classification_models[name] = pipe
            print(name, "Accuracy:", acc, "F1:", f1)

            cm = confusion_matrix(y_test, preds)
            disp = ConfusionMatrixDisplay(confusion_matrix=cm)
            disp.plot()
            plt.title(f"Confusion Matrix - {name}")
            plt.tight_layout()
            plt.savefig(f"{PLOT_DIR}/confusion_matrix_{name}.png", dpi=150)
            plt.close()

        classification_results_df = pd.DataFrame(classification_results)
        classification_results_df.to_csv(f"{TABLE_DIR}/classification_model_metrics.csv", index=False)
        display(classification_results_df)

        rf_pipe = fitted_classification_models.get("random_forest_classifier")
        if rf_pipe is not None:
            feature_names = get_feature_names(
                rf_pipe.named_steps["preprocessor"],
                cat_features,
                num_features
            )
            importances = rf_pipe.named_steps["model"].feature_importances_
            classification_feature_importance = (
                pd.DataFrame({"feature": feature_names[:len(importances)], "importance": importances})
                .sort_values("importance", ascending=False)
                .head(30)
            )
            classification_feature_importance.to_csv(f"{TABLE_DIR}/classification_feature_importance.csv", index=False)

            plt.figure(figsize=(10, 7))
            plot_df = classification_feature_importance.sort_values("importance", ascending=True).tail(20)
            plt.barh(plot_df["feature"], plot_df["importance"])
            plt.title("Classification Feature Importance - Same-Day Closure")
            plt.tight_layout()
            plt.savefig(f"{PLOT_DIR}/classification_feature_importance.png", dpi=150)
            plt.close()
    else:
        print("closed_same_day_flag has only one class. Skipping classification.")
else:
    print("closed_same_day_flag not found. Skipping classification.")


dummy_most_frequent Accuracy: 0.54208 F1: 0.0
logistic_regression Accuracy: 0.85392 F1: 0.8428030303030303
random_forest_classifier Accuracy: 0.8532 F1: 0.836277658815132


,model,target,rows_used,features_used,accuracy,precision,recall,f1
0,dummy_most_frequent,closed_same_day_flag,100000,"complaint_type, descriptor, borough, agency, a...",0.54208,0.000000,0.000000,0.000000
1,logistic_regression,closed_same_day_flag,100000,"complaint_type, descriptor, borough, agency, a...",0.85392,0.830788,0.855171,0.842803
2,random_forest_classifier,closed_same_day_flag,100000,"complaint_type, descriptor, borough, agency, a...",0.85320,0.854577,0.818746,0.836278



# Report Findings

This section creates a concise findings table that can be used directly in the Initial Findings Report and Executive Summary.


In [ ]:
# 17. Build Report-Ready Findings Table

findings = []

if "complaint_type" in full_count_tables and len(full_count_tables["complaint_type"]) > 0:
    top_row = full_count_tables["complaint_type"].iloc[0]
    findings.append({
        "finding_id": "F1",
        "topic": "Complaint concentration",
        "result": f"The most common complaint type is {top_row['complaint_type']} with {int(top_row['count']):,} records in the full dataset.",
        "evidence_file": "full_counts_complaint_type.csv",
        "business_relevance": "High-volume complaint types are the best starting point for operational improvement."
    })

if "borough" in full_count_tables and len(full_count_tables["borough"]) > 0:
    top_row = full_count_tables["borough"].iloc[0]
    findings.append({
        "finding_id": "F2",
        "topic": "Geographic distribution",
        "result": f"The borough with the highest request count is {top_row['borough']} with {int(top_row['count']):,} records in the full dataset.",
        "evidence_file": "full_counts_borough.csv",
        "business_relevance": "Uneven request distribution supports resource allocation analysis."
    })

if "response_time_hours" in df_sample.columns:
    rt = pd.to_numeric(df_sample["response_time_hours"], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    rt = rt[rt >= 0]
    if len(rt) > 0:
        findings.append({
            "finding_id": "F3",
            "topic": "Response time skew",
            "result": f"The median response time in the modeling sample is {rt.median():.2f} hours, while the 95th percentile is {rt.quantile(0.95):.2f} hours.",
            "evidence_file": "sample_response_time_summary.csv",
            "business_relevance": "The gap between median and upper-tail response time shows that delays are concentrated in a subset of cases."
        })

if 'classification_results_df' in globals() and len(classification_results_df) > 0:
    best = classification_results_df.sort_values("f1", ascending=False).iloc[0]
    findings.append({
        "finding_id": "F4",
        "topic": "Classification modeling",
        "result": f"The best same-day closure model was {best['model']} with F1 score {best['f1']:.3f}.",
        "evidence_file": "classification_model_metrics.csv",
        "business_relevance": "Classification results support early identification of requests likely to miss same-day resolution."
    })

if 'regression_results_df' in globals() and len(regression_results_df) > 0:
    best = regression_results_df.sort_values("rmse", ascending=True).iloc[0]
    findings.append({
        "finding_id": "F5",
        "topic": "Regression modeling",
        "result": f"The best response-time model was {best['model']} with RMSE {best['rmse']:.2f} hours.",
        "evidence_file": "regression_model_metrics.csv",
        "business_relevance": "Regression modeling helps estimate service delay risk using available request attributes."
    })

top_findings_table = pd.DataFrame(findings)
top_findings_table.to_csv(f"{TABLE_DIR}/top_findings_table.csv", index=False)
display(top_findings_table)


,finding_id,topic,result,evidence_file,business_relevance
0,F1,Complaint concentration,The most common complaint type is Illegal Park...,full_counts_complaint_type.csv,High-volume complaint types are the best start...
1,F2,Geographic distribution,The borough with the highest request count is ...,full_counts_borough.csv,Uneven request distribution supports resource ...
2,F3,Response time skew,The median response time in the modeling sampl...,sample_response_time_summary.csv,The gap between median and upper-tail response...
3,F4,Classification modeling,The best same-day closure model was logistic_r...,classification_model_metrics.csv,Classification results support early identific...
4,F5,Regression modeling,The best response-time model was random_forest...,regression_model_metrics.csv,Regression modeling helps estimate service del...


In [ ]:
# 18. Consolidated Model Metrics and Excel Workbook

model_metrics_frames = []

if 'regression_results_df' in globals() and len(regression_results_df) > 0:
    model_metrics_frames.append(regression_results_df)

if 'classification_results_df' in globals() and len(classification_results_df) > 0:
    model_metrics_frames.append(classification_results_df)

if model_metrics_frames:
    model_metrics = pd.concat(model_metrics_frames, ignore_index=True, sort=False)
else:
    model_metrics = pd.DataFrame()

model_metrics.to_csv(f"{TABLE_DIR}/model_metrics.csv", index=False)

excel_path = f"{TABLE_DIR}/analysis_summary_tables.xlsx"
with pd.ExcelWriter(excel_path) as writer:
    final_field_list.to_excel(writer, sheet_name="final_fields", index=False)
    missing_value_treatment.to_excel(writer, sheet_name="missing_treatment", index=False)
    derived_features_summary.to_excel(writer, sheet_name="derived_features", index=False)
    transformation_rules_applied.to_excel(writer, sheet_name="transformations", index=False)
    top_findings_table.to_excel(writer, sheet_name="top_findings", index=False)
    model_metrics.to_excel(writer, sheet_name="model_metrics", index=False)

    for name, tbl in full_count_tables.items():
        tbl.to_excel(writer, sheet_name=f"full_{name}"[:31], index=False)

    for name, tbl in summary_tables.items():
        tbl.to_excel(writer, sheet_name=name[:31], index=False)

print("Saved:", excel_path)


Saved: /content/project_outputs/03_analysis/tables/analysis_summary_tables.xlsx


#------------------Remove for post -------------------


# Completion Summary and Zip

This final step writes a plain-text completion summary and zips all Notebook 3 outputs.


In [ ]:
# 19. Completion Summary and Zip Outputs

completion_summary = f'''
NOTEBOOK 3 COMPLETE

What this notebook did:
1. Loaded Notebook 2 handoff files from notebook2_outputs.zip
2. Loaded the analysis-ready modeling sample
3. Resolved the full source parquet path from the manifest
4. Ran full-dataset batch aggregations without loading all 20M rows into pandas
5. Created descriptive analysis tables and charts
6. Ran sample-based response-time regression models
7. Ran sample-based same-day closure classification models
8. Exported model metrics and feature importance
9. Created a top findings table for report writing
10. Created an Excel workbook of analysis summary tables
11. Zipped all Notebook 3 outputs

Important scaling note:
- Full 20M data was used for batch aggregation only.
- Modeling was performed on the prepared sample from Notebook 2.
- This is intentional for standard Colab RAM limits.

Primary outputs:
- analysis_summary_tables.xlsx
- model_metrics.csv
- regression_model_metrics.csv
- classification_model_metrics.csv
- top_findings_table.csv
- full_counts_*.csv
- plots/*.png

Output directory:
{ANALYSIS_DIR}
'''.strip()

summary_path = f"{ANALYSIS_DIR}/notebook3_completion_summary.txt"
with open(summary_path, "w") as f:
    f.write(completion_summary)

zip_path = "/content/notebook3_outputs.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive("/content/notebook3_outputs", "zip", ANALYSIS_DIR)

print(completion_summary)
print("\nCreated zip:", zip_path)


NOTEBOOK 3 COMPLETE

What this notebook did:
1. Loaded Notebook 2 handoff files from notebook2_outputs.zip
2. Loaded the analysis-ready modeling sample
3. Resolved the full source parquet path from the manifest
4. Ran full-dataset batch aggregations without loading all 20M rows into pandas
5. Created descriptive analysis tables and charts
6. Ran sample-based response-time regression models
7. Ran sample-based same-day closure classification models
8. Exported model metrics and feature importance
9. Created a top findings table for report writing
10. Created an Excel workbook of analysis summary tables
11. Zipped all Notebook 3 outputs

Important scaling note:
- Full 20M data was used for batch aggregation only.
- Modeling was performed on the prepared sample from Notebook 2.
- This is intentional for standard Colab RAM limits.

Primary outputs:
- analysis_summary_tables.xlsx
- model_metrics.csv
- regression_model_metrics.csv
- classification_model_metrics.csv
- top_findings_table.csv
-

#------------------End Remove for post ---------------

In [ ]:
zip_path = "/content/notebook3_outputs.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive("/content/notebook3_outputs", "zip", ANALYSIS_DIR)

print(completion_summary)
print("\nCreated zip:", zip_path)


## Notebook 3 Final Summary

The zipped handoff file for report writing is:

- `notebook3_outputs.zip`

Upload that zip file for the final report and Executive Summary drafting.
